# CLIP-Augmented Distillation — Variant 1: Weighted Soft-Target Blend

**Branch:** `feature/clip-distillation-teacher` (Option A from the FYP CLIP integration plan)

**Claim.** Adding CLIP's zero-shot predictions as a *second, training-time-only* teacher —
blended with the existing EfficientNet-B3 teacher's soft targets before computing the
distillation loss — can correlate the image classification signal with CLIP's broader
visual-semantic knowledge, without adding a single parameter to the deployed model. The
student is still plain MobileNetV3-Large (3.4M params); CLIP (151M params) is discarded
after training.

**Alternatives seriously considered.**
- *Separate dual-KD loss terms* instead of blending — implemented in the companion notebook
  `06_clip_distillation_dual_kd.ipynb`. That notebook and this one are a deliberately
  controlled pair: both use `ALPHA = 0.7` and `BLEND_WEIGHT = 0.3`, so the *only* difference
  between their results is blend-before-KL vs. two separate KL terms, not a difference in
  how much total weight CLIP received.
- *Replacing* the EfficientNet-B3 teacher with CLIP entirely — rejected because EfficientNet-B3
  was fine-tuned on this exact dataset (73.65% val acc) and is a strictly stronger *specialist*
  signal than CLIP's zero-shot generalist one; discarding it would very likely hurt rather than
  help.

**Rejection criteria for this variant.** If this notebook's test accuracy comes in below the
baseline's, blending is rejected as an integration strategy in favor of the dual-KD variant (or
CLIP-as-teacher is rejected outright if *both* variants underperform the baseline).

**Anticipated attack, answered up front.** *"You're averaging two probability distributions
before computing one KL divergence — doesn't that discard exactly the information you'd want,
i.e. the *disagreement* between the two teachers?"* Yes. When EfficientNet and CLIP strongly
disagree on an image, their blended target can land on a class neither teacher actually
favored (e.g. 0.9/0.1 and 0.1/0.9 averages to a mushy 0.5/0.5). The dual-KD notebook exists
specifically to test whether preserving that disagreement (as two separate loss pulls) changes
the outcome — treat the two notebooks together as the actual experiment, not this one in
isolation.

**A second, more subtle simplification, also worth stating rather than leaving for a panelist
to find**: CLIP's own zero-shot probabilities are already calibrated through its own learned
`logit_scale` (its native temperature, tuned during CLIP's own pretraining). The existing
EfficientNet distillation uses `TEMPERATURE = 4` to soften *its* raw classification logits —
a value tuned for EfficientNet's logit scale, not CLIP's cosine-similarity scale. Blending the
two into one target and then comparing it against a single `T=4`-softened student distribution
means CLIP's contribution is implicitly re-tempered by a temperature that was never tuned for
it. This is a known limitation of the blend approach and, again, part of why the dual-KD
variant exists — there, the CLIP term compares against the student's own untouched (`T=1`)
distribution instead, avoiding the mismatch.

**Assumes** `03_model_training.ipynb` has already been run, producing
`../models/teacher_final.pth` and `../models/student_large_distilled.pth`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.metrics import classification_report
import open_clip

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────
CLASS_NAMES = [
    "Vitiligo", "Melasma", "Psoriasis", "Eczema",
    "Tinea", "Contact Dermatitis", "Seborrheic Dermatitis",
]  # order must match the numeric_label encoding in data/processed/*.csv

IMG_SIZE = 224          # student input size
B3_SIZE = 300           # EfficientNet-B3 teacher input size (matches 03_model_training.ipynb)

CLIP_MODEL_NAME = "ViT-B-32-quickgelu"  # "-quickgelu" matches the activation the "openai" weights were trained with
CLIP_PRETRAINED = "openai"

TEMPERATURE = 4         # unchanged from the baseline distillation_loss in 03_model_training.ipynb
ALPHA = 0.7             # unchanged — total non-hard-label weight, same as baseline
BLEND_WEIGHT = 0.3      # w — how much of the combined teacher signal comes from CLIP

TEACHER_CHECKPOINT = "../models/teacher_final.pth"
BASELINE_STUDENT_CHECKPOINT = "../models/student_large_distilled.pth"  # no-CLIP baseline, from 03_model_training.ipynb

TRAIN_CSV = "../data/processed/train.csv"
VAL_CSV = "../data/processed/val.csv"
TEST_CSV = "../data/processed/test.csv"

## Loading the frozen EfficientNet-B3 teacher
Identical architecture and checkpoint to `03_model_training.ipynb` — this teacher is not retrained here, only used to produce soft targets.

In [ ]:
# ── EfficientNet-B3 Teacher (identical architecture to 03_model_training.ipynb) ──
def build_teacher_model(num_classes=7):
    model = models.efficientnet_b3(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.SiLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes)
    )
    return model

teacher = build_teacher_model(num_classes=len(CLASS_NAMES))
teacher.load_state_dict(torch.load(TEACHER_CHECKPOINT, map_location=device, weights_only=True))
teacher = teacher.to(device)
teacher.eval()
for param in teacher.parameters():
    param.requires_grad = False

print(f"EfficientNet-B3 teacher loaded from {TEACHER_CHECKPOINT}, frozen ({sum(p.numel() for p in teacher.parameters()):,} params)")

## Loading CLIP as a second, zero-shot teacher
CLIP never sees a single gradient update here or in deployment — `requires_grad = False` on
every parameter. Its only job is to produce a soft probability distribution over the 7 classes
for each training image, using the same zero-shot prompting approach as
`inference/server.py`'s `/predict_online` endpoint (branch `feature/clip-online-secondary-opinion`).

In [ ]:
print(f"Loading CLIP {CLIP_MODEL_NAME} ({CLIP_PRETRAINED})...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED)
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
clip_model = clip_model.to(device)
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

clip_param_count = sum(p.numel() for p in clip_model.parameters())
print(f"CLIP loaded and frozen ({clip_param_count:,} params) — training-time only, never ships to the device.")

In [ ]:
# Kept in sync with inference/server.py's CLIP_PROMPTS (branch
# feature/clip-online-secondary-opinion) for the 7 real classes. "No Disease" and
# "Unknown" are omitted here on purpose — training labels only cover these 7 classes,
# so there is no ground truth to distill an open-set signal against at training time.
CLIP_PROMPTS = {
    "Vitiligo": [
        "a photo of vitiligo, depigmented white patches on the skin",
        "a dermatology image of skin with irregular white patches from loss of pigment",
        "a close-up of skin showing patchy loss of melanin",
    ],
    "Melasma": [
        "a photo of melasma, brown or gray-brown patches on the face",
        "a dermatology image of facial skin hyperpigmentation in symmetric patches",
        "a close-up of blotchy, darkened skin discoloration on the cheeks or forehead",
    ],
    "Psoriasis": [
        "a photo of psoriasis, red scaly plaques on the skin",
        "a dermatology image of thick, silvery-scaled red skin lesions",
        "a close-up of inflamed, flaky raised skin patches",
    ],
    "Eczema": [
        "a photo of eczema, dry inflamed and itchy red skin",
        "a dermatology image of atopic dermatitis with cracked, irritated skin",
        "a close-up of red, scaly, inflamed skin patches",
    ],
    "Tinea": [
        "a photo of tinea, a ring-shaped fungal skin infection",
        "a dermatology image of a red, scaly, circular rash with a clear center",
        "a close-up of ringworm-like fungal skin lesion",
    ],
    "Contact Dermatitis": [
        "a photo of contact dermatitis, red irritated skin from an allergic reaction",
        "a dermatology image of inflamed skin with redness and swelling from irritation",
        "a close-up of a rash caused by skin contact with an irritant or allergen",
    ],
    "Seborrheic Dermatitis": [
        "a photo of seborrheic dermatitis, greasy yellowish scaly patches on the skin",
        "a dermatology image of flaky, oily skin inflammation",
        "a close-up of red skin with greasy yellow scales",
    ],
}
assert list(CLIP_PROMPTS.keys()) == CLASS_NAMES, "CLIP_PROMPTS keys must match CLASS_NAMES order exactly"
print(f"{len(CLASS_NAMES)} classes, {sum(len(v) for v in CLIP_PROMPTS.values())} total prompt variants")

**What it does:** embeds all prompt variants per class, averages them into one embedding per class (prompt ensembling), and caches the result — this only needs to happen once, not per training image.

In [ ]:
def _embed_clip_text(prompts):
    tokens = clip_tokenizer(prompts).to(device)
    with torch.no_grad():
        embeddings = clip_model.encode_text(tokens)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
    return embeddings

with torch.no_grad():
    _class_embeddings = []
    for class_name in CLASS_NAMES:
        variant_embeddings = _embed_clip_text(CLIP_PROMPTS[class_name])
        class_embedding = variant_embeddings.mean(dim=0)
        class_embedding = class_embedding / class_embedding.norm()
        _class_embeddings.append(class_embedding)
    CLIP_TEXT_EMBEDDINGS = torch.stack(_class_embeddings).to(device)  # [7, embed_dim]

print(f"Cached {CLIP_TEXT_EMBEDDINGS.shape[0]} per-class CLIP text embeddings (prompt-ensembled)")

## Dataset
Standard single-transform dataset, reused for the CLIP-precompute pass, the teacher-alone eval, and loading the baseline student.

In [ ]:
class SkinDataset(Dataset):
    """Same as 03_model_training.ipynb's SkinDataset — single transform, (image, label)."""
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = row['numeric_label']
        try:
            image = Image.open(row['image_path']).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224))
        if self.transform:
            image = self.transform(image)
        return image, label

### Why CLIP is precomputed once, not re-run every epoch
The EfficientNet-B3 teacher **must** be re-run every epoch in this pipeline, because its input
(`b3_train_transform`) includes random augmentation (flip/rotate/color-jitter) — a fresh random
view every time, so its soft target genuinely changes epoch to epoch. CLIP, by contrast, is fed
through its own standard zero-shot preprocessing (`clip_preprocess`: deterministic resize/crop/
normalize, no augmentation) — its output for a given training image is exactly the same on
epoch 1 as on epoch 40. There is nothing to gain and 40x the compute to lose by recomputing it
every epoch, so we run CLIP over each split **once**, cache the resulting probabilities, and the
training loop below just looks them up by index.

This is also a deliberate, defensible modeling choice, not just an optimization: it means CLIP
is queried with the *exact* zero-shot preprocessing it was pretrained to expect, rather than
our custom augmentation pipeline (which CLIP was never trained on).

In [ ]:
@torch.no_grad()
def precompute_clip_probs(csv_path, batch_size=64):
    """Runs CLIP once over an entire split and returns a [N, 7] tensor of zero-shot
    class probabilities, in the same row order as csv_path. See markdown above for
    why this is safe to precompute rather than recompute every epoch."""
    dataset = SkinDataset(csv_path, transform=clip_preprocess)
    # num_workers=0: classes defined in a live notebook cell (SkinDataset here) aren't
    # reliably importable by spawned worker processes on native Windows — num_workers>0
    # raises "module '__main__' has no attribute 'SkinDataset'" in that environment. If
    # you're on Linux/Colab, num_workers=4 (matching 03_model_training.ipynb) is safe and faster.
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    all_probs = []
    for images, _ in loader:
        images = images.to(device)
        image_features = clip_model.encode_image(images)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        logits = clip_model.logit_scale.exp() * image_features @ CLIP_TEXT_EMBEDDINGS.T
        probs = F.softmax(logits, dim=-1)
        all_probs.append(probs.cpu())

    return torch.cat(all_probs, dim=0)

print("Precomputing CLIP zero-shot probabilities for train/val/test (one pass each)...")
clip_probs_train = precompute_clip_probs(TRAIN_CSV)
clip_probs_val = precompute_clip_probs(VAL_CSV)
clip_probs_test = precompute_clip_probs(TEST_CSV)
print(f"train: {clip_probs_train.shape} | val: {clip_probs_val.shape} | test: {clip_probs_test.shape}")

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

b3_train_transform = transforms.Compose([
    transforms.Resize((B3_SIZE, B3_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

b3_val_transform = transforms.Compose([
    transforms.Resize((B3_SIZE, B3_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Transforms defined (student 224px augmented, teacher 300px augmented, CLIP uses its own clip_preprocess)")

In [ ]:
train_labels = pd.read_csv(TRAIN_CSV)['numeric_label'].values
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
print(f"Class counts: {class_counts}")
print("Weighted sampler built (same inverse-frequency scheme as 03_model_training.ipynb)")

## Assembling the distillation loader
Each batch now yields four things instead of three: the student's augmented 224px view, the teacher's augmented 300px view, CLIP's precomputed probabilities for that row, and the label.

In [ ]:
class DistillDataset(Dataset):
    """Returns (student_img, teacher_img, clip_probs, label). clip_probs is looked up
    from the precomputed tensor by row index — no CLIP forward pass happens here."""
    def __init__(self, csv_path, student_transform, teacher_transform, clip_probs):
        self.df = pd.read_csv(csv_path)
        self.student_transform = student_transform
        self.teacher_transform = teacher_transform
        self.clip_probs = clip_probs

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = row['numeric_label']
        try:
            image = Image.open(row['image_path']).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224))
        return (
            self.student_transform(image),
            self.teacher_transform(image),
            self.clip_probs[idx],
            label,
        )

distill_dataset = DistillDataset(TRAIN_CSV, train_transform, b3_train_transform, clip_probs_train)
# num_workers=0 here too — see the comment on precompute_clip_probs's loader above.
distill_loader = DataLoader(distill_dataset, batch_size=16, sampler=sampler, num_workers=0, pin_memory=True)

val_dataset = SkinDataset(VAL_CSV, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

test_dataset = SkinDataset(TEST_CSV, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"Distillation loader: {len(distill_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")

In [ ]:
def val_epoch(model, loader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100. * correct / total

## The blended distillation loss

`combined_probs` mixes the EfficientNet teacher's `T`-softened distribution with CLIP's
already-calibrated distribution using a single scalar, `BLEND_WEIGHT` (w = 0.3 by default).
At `w = 0` this is byte-for-byte the baseline `distillation_loss` from
`03_model_training.ipynb` — that is the entire point of using a blend weight instead of a
separate ablation script: the "without CLIP" arm of this ablation isn't a different
notebook, it's this same function with one number changed.

In [ ]:
def blended_kd_loss(student_logits, teacher_logits, clip_probs, labels,
                     temperature=TEMPERATURE, alpha=ALPHA, blend_weight=BLEND_WEIGHT):
    # EfficientNet teacher's own distribution, softened the same way as the baseline.
    soft_teacher = F.softmax(teacher_logits / temperature, dim=1)

    # CLIP's distribution is already calibrated via its own logit_scale — blended in
    # as-is (see markdown above for why we don't additionally apply `temperature` to it).
    combined_probs = (1 - blend_weight) * soft_teacher + blend_weight * clip_probs

    soft_student = F.log_softmax(student_logits / temperature, dim=1)
    distill_loss = F.kl_div(soft_student, combined_probs, reduction='batchmean') * (temperature ** 2)

    hard_loss = F.cross_entropy(student_logits, labels, label_smoothing=0.1)

    return alpha * distill_loss + (1 - alpha) * hard_loss

print(f"Blended KD loss defined | Temperature: {TEMPERATURE} | Alpha: {ALPHA} | Blend weight (CLIP share): {BLEND_WEIGHT}")

In [ ]:
def build_student_large(num_classes=7):
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = True
    in_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.Hardswish(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes)
    )
    return model

student = build_student_large(num_classes=len(CLASS_NAMES))
student = student.to(device)
print(f"Student (MobileNetV3-Large) params: {sum(p.numel() for p in student.parameters()):,}")

In [ ]:
student_optimizer = torch.optim.Adam(student.parameters(), lr=0.0001, weight_decay=1e-4)
student_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(student_optimizer, mode='min', patience=3, factor=0.5)
print("Optimizer and scheduler defined (identical hyperparameters to the baseline run)")

In [ ]:
# ── Distillation Training Loop (blended teacher) ──────────────────────
def train_distillation_epoch(student, teacher, loader, optimizer):
    student.train()
    teacher.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for student_imgs, teacher_imgs, clip_probs, labels in loader:
        student_imgs = student_imgs.to(device)
        teacher_imgs = teacher_imgs.to(device)
        clip_probs = clip_probs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        student_logits = student(student_imgs)

        with torch.no_grad():
            teacher_logits = teacher(teacher_imgs)

        loss = blended_kd_loss(student_logits, teacher_logits, clip_probs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = student_logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100. * correct / total


NUM_EPOCHS = 40
best_val_loss = float('inf')
patience = 5
patience_counter = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_distillation_epoch(student, teacher, distill_loader, student_optimizer)
    val_loss, val_acc = val_epoch(student, val_loader)

    student_scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(student.state_dict(), '../models/student_large_clip_blend_distilled.pth')
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% \u2713 saved")
    else:
        patience_counter += 1
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

print("\nCLIP-blend distillation complete!")

## Evaluation
Load the best checkpoint saved above, and evaluate it — plus the baseline (no-CLIP) student, CLIP alone, and the EfficientNet teacher alone — on the same held-out test set, in this same notebook session, so all four numbers are directly comparable.

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_labels, all_preds

In [ ]:
student.load_state_dict(torch.load('../models/student_large_clip_blend_distilled.pth', map_location=device, weights_only=True))
new_labels, new_preds = evaluate(student, test_loader)
print("── CLIP-blend student on the test set ──")
print(classification_report(new_labels, new_preds, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
baseline_student = build_student_large(num_classes=len(CLASS_NAMES)).to(device)
baseline_student.load_state_dict(torch.load(BASELINE_STUDENT_CHECKPOINT, map_location=device, weights_only=True))

baseline_labels, baseline_preds = evaluate(baseline_student, test_loader)
print(f"── Baseline student ({BASELINE_STUDENT_CHECKPOINT}, NO CLIP) on the test set ──")
print(classification_report(baseline_labels, baseline_preds, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
clip_test_preds = clip_probs_test.argmax(dim=1).numpy()
clip_test_labels = pd.read_csv(TEST_CSV)['numeric_label'].values
print("── CLIP zero-shot ALONE on the test set (diagnostic — CLIP never sees fine-tuning) ──")
print(classification_report(clip_test_labels, clip_test_preds, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
test_dataset_b3 = SkinDataset(TEST_CSV, transform=b3_val_transform)
test_loader_b3 = DataLoader(test_dataset_b3, batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

teacher_labels, teacher_preds = evaluate(teacher, test_loader_b3)
print("── EfficientNet-B3 teacher ALONE on the test set (for reference) ──")
print(classification_report(teacher_labels, teacher_preds, target_names=CLASS_NAMES, zero_division=0))

## Benchmark table row
Formatted to drop straight into the shared benchmark table from `CONTEXT.md`.

In [ ]:
from sklearn.metrics import accuracy_score

new_labels, new_preds = evaluate(student, test_loader)

rows = [
    {"variant": "Baseline (no CLIP, from 03_model_training.ipynb)",
      "checkpoint": BASELINE_STUDENT_CHECKPOINT,
      "on_device_params": sum(p.numel() for p in student.parameters()),
      "test_accuracy": accuracy_score(baseline_labels, baseline_preds)},
    {"variant": "CLIP-blend distillation (this notebook, w=0.3)",
      "checkpoint": "../models/student_large_clip_blend_distilled.pth",
      "on_device_params": sum(p.numel() for p in student.parameters()),
      "test_accuracy": accuracy_score(new_labels, new_preds)},
    {"variant": "CLIP zero-shot alone (diagnostic, not deployed)",
      "checkpoint": "n/a (no training)",
      "on_device_params": 0,
      "test_accuracy": accuracy_score(clip_test_labels, clip_test_preds)},
    {"variant": "EfficientNet-B3 teacher alone (reference)",
      "checkpoint": TEACHER_CHECKPOINT,
      "on_device_params": 0,
      "test_accuracy": accuracy_score(teacher_labels, teacher_preds)},
]

benchmark_df = pd.DataFrame(rows)
benchmark_df["test_accuracy"] = (benchmark_df["test_accuracy"] * 100).round(2)
print(benchmark_df.to_string(index=False))
print()
print("Copy the 'Baseline' and 'CLIP-blend distillation (this notebook, w=0.3)' rows into the shared benchmark table")
print("(image-only accuracy row) from CONTEXT.md — on-device params are identical")
print("between them since CLIP never ships to the device.")

## Summary for the defense

- **On-device cost: unchanged.** The deployed student is still the same 3.4M-parameter
  MobileNetV3-Large — CLIP (151M params) and EfficientNet-B3 exist only in this notebook,
  never in `inference/server.py`.
- **What changed:** the *training signal* the student learns from now includes CLIP's
  zero-shot visual-semantic judgment, blended with the existing specialist teacher via one
  interpretable knob (`BLEND_WEIGHT`).
- **Compare this notebook's row against `06_clip_distillation_dual_kd.ipynb`'s row** — same
  `ALPHA`/`BLEND_WEIGHT` budget, different mechanism (blend vs. separate KD terms). Whichever
  wins (or if neither beats baseline) is itself a reportable finding, not just a means to
  picking a shipped model.